In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧱 W15-D1 · LnkChatBI 架构精读①：NL→SQL 组装链路

> 开发期 · Week 15「LnkChatBI 精读 × Semantic Model 第一个消费方」Day 1（2026-09-07 周一）
> 昨日 D7 Virtual CTO Review 已裁决 W15 双线 GO：LnkChatBI 精读 + term-aliases 第一个消费方，附四条护栏（指纹锚 / 场景层冻结 / 名称规范 / Demo 前先测导入前基线）。

---


## 0. 今日开发目标

精读 `/root/LnkChatBI`（五件套中唯一未进过学习计划的仓库）的 **NL→SQL 组装链路**，走完四段：chat streaming 契约 → prompt 组装 → SQL 生成 → 权限下推；再把 PT-W4-D6 的 **L1-L5 验证标准**翻译成 LnkChatBI 语境下**可机器复核的 Demo 验收口径**，为 D6 双 Demo（术语库导入 + A101 依据链问答）定好靶子。

---


## 1. Today's Question：为什么第一个消费方选"问答"而不是"生成代码"或"自动审批"？

**一句话答案：三个消费方里，只有"问答"同时满足四个条件——失败成本被只读闸门锁死在零、恰好只吃 Semantic Model v0.1 唯一完整的面（术语层）、每一次错误回答都能沉淀为校准示例形成反哺回路、验收证据全部落在 chat record log 里可机器复核。D7 评审说了前半句（"问答恰好只吃术语层"），今天代码精读补齐了后半句：连"锁死在零"这件事本身，都是链路里一行行代码保证的。**

### 1.1 先看链路本身（今日实测，全部来自真实代码）

`run_task()`（`apps/chat/task/llm.py:1761`）是一条**九级流水**，每一级都有 OperationEnum 落库 + SSE 事件外发：

| 级 | 代码事实 | OperationEnum / SSE 事件 |
|---|---|---|
| ① RAG 三件套检索 | `filter_terminology_template`（术语：PG `ILIKE '%'\|\|word\|\|'%'` 子串匹配 + 可选 pgvector 向量双路）→ `filter_training_template`（SQL 示例校准集）→ `filter_custom_prompts`（自定义提示） | FILTER_TERMS / FILTER_SQL_EXAMPLE / FILTER_CUSTOM_PROMPT（本地操作，无 LLM） |
| ② 表结构选择 | `choose_table_schema` → `get_table_schema` 产出 **M-Schema** 格式（表注释 `custom_comment` + 字段注释），`TABLE_EMBEDDING_ENABLED` 时按问题相关性选表 | CHOOSE_TABLE（本地操作） |
| ③ 消息组装 | `init_messages`：System prompt（XML 标签体系：`<Instruction>`/`<m-schema>`/`<terminologies>`/`<sql-examples>`/`<Other-Infos>`/`<Rules>`/`<SQL-Generation-Process>` 九步检查）+ 历史轮次（`base_message_round_count_limit` 截断） | — |
| ④ 契约启动 | SSE 首批事件；无数据源时 `select_datasource` 用 LLM 从数据源列表选库 | `id` → `question`（→ `datasource-result` 流式 → `datasource`） |
| ⑤ SQL 生成 | `generate_sql` 流式生成；LLM 按 JSON 契约返回 `{success, sql, tables, chart-type, brief}` | GENERATE_SQL / `sql-result`（逐 token）→ `info` → `brief` |
| ⑥ SQL 校验 | `check_sql`：JSON 解析 + **fallback 裸 SQL 提取**（```sql 围栏或 SELECT/WITH 开头，单语句 + 只读安全判定才收） | 返回 `(sql, tables)` |
| ⑦ 权限下推 | `is_normal_user` → `get_row_permission_filters`（row 权限 `expression_tree` → `transFilterTree` → WHERE 串）→ 有过滤则**二次 LLM 调用** `build_table_filter`（permissions 模板把 WHERE 合并进 SQL）→ `check_save_sql` 复验 | GENERATE_SQL_WITH_PERMISSIONS / `sql`（格式化终版） |
| ⑧ 执行 | `exec_sql` → `check_sql_read`：**sqlglot AST 解析**，Insert/Update/Delete/Create/Drop/Alter/Merge/Command 九类写操作黑名单，命中即拒 | EXECUTE_SQL / `sql-data` |
| ⑨ 图表 | `used_tables_schema`（只含实际用过的表）二次注入 → `generate_chart` | GENERATE_CHART / `chart-result` → `chart` → `finish` |

**SSE 事件契约**（`apps/chat/streaming/events.py` + 前端 `ChartAnswer.vue` 实测对齐）：`id / regenerate_record_id / question / info / brief / error / sql-result / sql / sql-data / chart-result / chart / datasource / finish`，统一 `emit_chat_event` 序列化（`data:{json}\n\n`）。openspec `chat-streaming-contract` 规格明确四条：共享序列化路径、错误事件统一形状、finish 只在编排层终态后发、前端走共享适配器消费——**契约是规格管着的，不是约定俗成**。

### 1.2 回答 Today's Question：四层论证

**① 失败成本被代码锁死在零（只读闸门是确定性的）。** 问答消费方的全部产出是 SELECT。`check_sql_read` 用 sqlglot 把 SQL 解析成 AST 再做类型黑名单判断——不是正则、不是 prompt 里求 LLM"请别写坏"（prompt 规则 #2 也说了"只能生成查询用 SQL"，但真正拦住的是 AST 这一层）。错了最坏是一次错误回答，用户追问一句就纠正。对比：生成代码的错误进编译器和运行时，自动审批的错误直接造成资金/法律后果。**消费方选型第一问不是"AI 能做什么"而是"AI 错了的兜底在哪一层"**——问答的兜底是 AST，代码的兜底是测试和 review（成本高一个量级），审批的兜底是对账（月级滞后）。

**② 恰好只吃 v0.1 唯一完整的面。** 六构件里，Entity/术语层是唯一带度量的完整面（883 术语 + 别名 + 四家溯源）；Rule 层刚对完账（D4：registry 无代码锚点等四缺口登记在案）、Policy 层还没显式化（W15-D4 才做语义化）。问答消费恰好只要求：听懂术语（`terminologies` XML 块注入）+ 表结构（M-Schema）+ SQL 组装能力。生成代码要吃 Relationship/Rule 的谓词语义，自动审批要吃 Policy/Lifecycle——**都在 v0.1 能力圈外**。让消费方等资产成熟，而不是让资产被不成熟的消费方烧掉信誉。

**③ 反哺回路：消费即校准。** LnkChatBI 的 `data_training`（SQL 示例校准）机制意味着每一次好的问答都能沉淀为 few-shot 示例、每一次术语命中都能验证术语库质量——**问答是唯一能把"消费"变成"反哺"的形态**（README 说"越问越准"，代码里就是 FILTER_SQL_EXAMPLE + FILTER_TERMS 两条本地操作）。生成代码和自动审批的消费行为不产生训练信号。

**④ 验收可判定。** 问答的证据全部落在 chat record log（sql / tables / sql-data / terminologies 块），L1-L5 判定梯可以逐级对着 log 机器复核（见 §2.2）。代码生成的验收要跑测试套件，审批的验收要构造审批流——Demo 成本都不在一个量级。

### 1.3 ERP 人话（26 年对照）

老会计都懂一条规矩：**先让实习生查数，别让他记账**。查数错了，对一眼就发现；记账错了，月底对账才能翻出来，翻出来已经是事故。26 年 ERP 里第一个 AI 消费方选"报表问答"而不是"自动过账"或"自动下单"，是同一个逻辑。LnkChatBI 的 AST 只读闸门，就是把"实习生只能看不能改"这条铁律写进了代码——而不是写在制度里。

---


## 2. 完成事实

### 2.1 Demo 验收口径（L1-L5 × LnkChatBI 证据源）——今日主交付

PT-W4-D6 的 L1-L5 判定梯（权重：L1 10% 必达 / L2 20% 必达 / L3 25% 必达 / L4 15% 加分 / L5 30% 超额），翻译到 LnkChatBI 语境：

| 级 | PT-W4 原判据 | LnkChatBI 落地判据（D6 Demo 靶子） | 机器复核证据源 |
|---|---|---|---|
| **L0 基线护栏**（D7 裁决追加） | — | 导入术语库**前**先跑一轮同题问答记录基线（没有基线的 Demo 是展示不是验证） | 导入前 record log |
| **L1 语义理解**（必达） | 身份路径 `Project→Building→Floor→A101`，非 status 字段串 | 问"A101 铺位…"：SQL 命中 `bi_d_position` 且谓词含 `POSITION_CODE='A101'`（种子码为 `LOC_DEMO_L1xx` 风格，需补种 A101 行或做别名映射——**这正是 term-aliases 的用武之地**）；结果行含 STORE/BUILDING/FLOOR/POSITION 四级路径列 | record log：`sql` + `tables` + `sql-data` |
| **L2 业务链推理**（必达） | 关系遍历 + 状态机节点 | SQL 沿 `CONT_NO` join 合同/租户表；空置判断引用 `POSITION_STATE`/`END_DATE`（种子已备好：L2-02 快闪铺 state=2、CONT_NO='-'、END_DATE=2099，是天然的"不能出租"素材），而非只报状态串 | record log：`sql` + 结果行 |
| **L3 规则判断**（必达，Text-to-SQL 形态降级） | Rule ID + 失败分支 | Text-to-SQL 不跑规则引擎，降级口径：规则以**术语 description** 注入（模板原文：description"可能是能够用来参考的计算公式，或者是一些其他的查询条件"）——答案需引用该条件方可判过。**这是 Semantic Model Rule 构件 → LnkChatBI 的天然接口** | FILTER_TERMS log（`terminologies` 块）+ 答案文本 |
| L4 Policy 判断（加分） | 审批路径裁决 | 超出本 Demo，标 TODO（W15-D4 Policy 语义化后 v0.2 再验） | — |
| L5 动作建议（超额） | capability ID + 执行姿态 | 超出本 Demo，标 TODO | — |
| **L1' 权限条件保留**（今日新增，见 §2.3） | — | normal user + project 行权限下问答，断言最终 SQL 文本含权限 WHERE 谓词 | record log：改写后 `sql` 文本 |

**通过线：L0 + L1 + L2 + L3 全过 = Demo 达标（对齐 PT-W4"L1-L3 是地板"）；L1' 作为结构性护栏单独记录，不达标不算 Demo 失败但要进 Gap 列表。**

### 2.2 MI context provider 集成线（代码实测）

`apps/datasource/crud/context_provider.py`：契约是外部系统实现 `GET <provider_url>`（Bearer token + `X-SSO-Username` 头），返回 `{"username", "attributes"}`；`attribute_mapping` 把 attributes 映射为 SystemVariable(type=user) 写入 `sys_user.system_variables`，被 `transTreeItem` 在解析 `value_type="variable"` 的 expression_tree 行权限时消费。TTL 300s 缓存；**失败语义是 stale-but-safe**（拉取失败静默返回 False，用旧值）。这条线是 MI→LnkChatBI 行权限的桥梁，D2 细读。

### 2.3 今日挖出的结构性发现：权限下推是概率性执行

精读 ⑦ 级发现三层叠加，值得单独立案：

1. **表清单来自 LLM 自报。** `check_sql` 返回的 `tables` 是 LLM JSON 里的自报字段；`get_row_permission_filters` 按 `tables` 查行权限——LLM 漏报一个 join 表，该表的行权限过滤**静默丢失**（无告警）。
2. **fallback 裸 SQL 路径 tables=None。** LLM 没返回 JSON 但返回了裸 SQL 时（```sql 围栏或 SELECT 开头 + 单语句 + 只读判定通过），`(fallback_sql, None)` → `get_row_permission_filters` 见空表列表直接返回 `[]` → **正常用户也整体跳过权限改写**，裸 SQL 原样执行。
3. **复验不含权限条件断言。** `check_save_sql` 只重跑 JSON/语法校验（+执行时 AST 只读闸门），**不验证改写后的 SQL 是否真的保留了 filter 谓词**——permissions 模板规则 #3"不要替换原来SQL中的过滤条件"只是 prompt 层约束。

**结论：这条链路里"只读"是确定性保证（AST），"行权限"是概率性执行（依赖 LLM 自报 + LLM 正确合并）。** 加固方向明确且便宜：表提取改走 AST/表注册对账（sqlglot 已是现成依赖），改写后加 filter 谓词存在性断言。已进 Gap 列表（候选：与 G-05 合并为主仓 change 提案），并落成验收口径的 L1' 护栏。

### 2.4 商业地产映射

mallcre 种子数据实测：`bi_d_position`（星河购物中心，4 铺位，含 1 个空置快闪铺）+ `bi_b_tenant`（3 租户），全库 570 张 `bi_cre_m3*` 明源血统表（SOURCE_NAME/UUID/FVERSION 审计列）。**"A101 为什么不能出租"在数据层已有现成素材**；缺的是让 LLM 听懂"A101"并知道去哪张表——即 Semantic Model 术语层，正是本周 D3 实验要生成的 term-aliases。

---


## 3. 遗留 / 风险

- **权限概率性执行**（§2.3）：D6 Demo 的 L1' 护栏会实测它；若确认可复现，与 G-05（effect-registry 冻结无 CI）、G-01（ontology 无 frontmatter）合并为向 LnkChatBI/主仓提 change 的候选包。
- 种子数据无字面 "A101"：D3 生成 term-aliases 时需含 `A101 → LOC_DEMO_*` 风格别名映射，或 D6 前补种——**别名映射更符合"术语层解决命名漂移"的定位**（D2 对账结论的直接应用）。
- v0.1.1 三件套（D7 整改）仍挂账在 D3 前置，未动。
- 今天只精读了链路主干；terminology embedding 双路检索的 pgvector 细节、assistant 动态数据源（type=1）分支留 D2。


## 4. 明日连接（D2 · LnkChatBI 架构精读②）

RAG 三件套（term-aliases / SQL 示例校准 / custom-prompt）+ MI context provider 集成线细读；对照 Orchestrator NL→SQL，划清 **Semantic Model 与 Text-to-SQL 的分工边界**——今天已埋下最重要的线索：术语 description 就是 Rule 构件的注入接口（L3 降级口径）。Today's Question：**Semantic Model 和 Text-to-SQL 的分界线到底在哪？**

---

### 附：今日证据清单

| 证据 | 来源 |
|---|---|
| 九级流水 / 事件顺序 / 权限下推二次 LLM 调用 / fallback 路径 / 表自报 | `apps/chat/task/llm.py` run_task:1761 / check_sql:1546 / build_table_filter:1418 / generate_filter:1469 / _extract_sql_fallback_candidate:246 |
| SSE 事件契约 + 前端消费 | `apps/chat/streaming/events.py` + `frontend/src/views/chat/answer/ChartAnswer.vue` + openspec `chat-streaming-contract/spec.md` |
| 只读 AST 闸门（九类写操作黑名单） | `apps/db/db.py` check_sql_read:1068 / exec_sql:863 |
| 行权限 expression_tree → WHERE + 用户/规则联查 | `apps/datasource/crud/permission.py` get_row_permission_filters:46 |
| MI context provider 契约（Bearer + X-SSO-Username → attributes → SystemVariable，TTL 300s，stale-but-safe） | `apps/datasource/crud/context_provider.py` + alembic 075 + `test_context_provider.py` |
| prompt XML 标签体系 / 术语 description 定位（"计算公式/查询条件"）/ 数据量零容忍规则 / permissions 模板规则 #3 | `backend/templates/template.yaml`（sql/terminology/data_training/permissions 四段原文） |
| 术语双路检索（ILIKE 子串 + pgvector） | `apps/terminology/crud/terminology.py` select_terminology_by_word:913 |
| M-Schema 组装 + 表 embedding 选表 | `apps/datasource/crud/datasource.py` get_table_schema:898 |
| 种子数据（星河购物中心 4 铺位含 1 空置 / 3 租户 / 570 表） | `mallcre_pg_init/mallcre_postgres.sql` + `mallcre_seed_realistic.sql` |
| L1-L5 权重与判据 | PT-W4-D6《组装SemanticModel与验证设计》判分表 |

*配套实验：`第15周-Day1-NL到SQL组装链路.ipynb` —— SSE 契约不变量校验器（3 条轨迹模拟）、权限下推概率性蒙特卡洛（漏报/fallback 双通路）、表注册对账确定性提取对照、L1-L3 验收判定器 v0.1（4 条样例轨迹判分）。*
